# PSU SLF AI — RAG Test (OCR + FAISS) บน Colab

โน้ตบุ๊กนี้จัดรูปแบบจากสคริปต์เดิมที่เคยใช้ทดสอบใน Colab (OCR รองรับ PDF สแกน + FAISS vector store +
retry เวลา Gemini error + โหมดถาม-ตอบแบบ interactive) โดยปรับ logic การค้นหาให้ตรงกับระบบจริงปัจจุบัน:
**ดึงคำตอบจากไฟล์ที่เกี่ยวข้องที่สุดเพียงไฟล์เดียว ไม่ปนหลายไฟล์ และไม่มี logic แยกผู้กู้รายเก่า/รายใหม่แล้ว**

**อย่าฝัง API key ลงในโค้ดตรงๆ** — เซลล์ด้านล่างใช้ `getpass` ให้กรอกตอนรันแทน เพื่อไม่ให้ key หลุดติดไปกับไฟล์/ประวัติแชท

In [ ]:
# ── 1. ติดตั้งไลบรารี + โปรแกรมระบบที่ OCR ต้องใช้ ──
!pip install -q google-genai langchain langchain-huggingface langchain-text-splitters \
    langchain-community sentence-transformers pypdf pymupdf faiss-cpu pdf2image pytesseract

# poppler-utils: แปลง PDF เป็นรูปภาพ (pdf2image) / tesseract-ocr + pack ภาษาไทย: อ่านตัวอักษรจากรูป
!apt-get -qq install -y poppler-utils tesseract-ocr tesseract-ocr-tha > /dev/null

In [ ]:
# ── 2. Imports ──
import os
import time
import getpass
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from google import genai
from pdf2image import convert_from_path
import pytesseract

In [ ]:
# ── 3. Gemini Configuration ──
# ขอ API key ฟรีได้ที่ https://aistudio.google.com/apikey
GEMINI_API_KEY = getpass.getpass('วาง Gemini API Key แล้วกด Enter: ')

client = genai.Client(api_key=GEMINI_API_KEY.strip())

# ทดสอบการเชื่อมต่อ Gemini
test = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='ทดสอบระบบ AI'
)
print('✅ Gemini Status:', test.text.strip())

In [ ]:
# ── 4. Upload PDF (เลือกได้หลายไฟล์พร้อมกัน) ──
print('📌 กรุณาเลือกไฟล์ PDF (สามารถเลือกพร้อมกันได้หลายไฟล์)')
uploaded = files.upload()

if not uploaded:
    raise Exception('ไม่พบไฟล์ PDF')

pdf_files = list(uploaded.keys())
print(f'\n✅ อัปโหลดสำเร็จทั้งหมด {len(pdf_files)} ไฟล์:')
for f in pdf_files:
    print(' -', f)

In [ ]:
# ── 5. PDF + OCR Loader (ใช้ text layer ก่อน ถ้าน้อยเกินไปค่อย OCR) ──
def load_single_pdf(pdf_path):
    documents = []
    file_name = os.path.basename(pdf_path)

    # 1. ลองดึง text layer ตรงจาก PDF ก่อน
    loader = PyPDFLoader(pdf_path)
    pdf_docs_from_text_layer = loader.load()

    text_layer_content_by_page = {}
    for doc in pdf_docs_from_text_layer:
        page_num = doc.metadata.get('page', 0)
        if doc.page_content.strip():
            text_layer_content_by_page[page_num] = doc.page_content.strip()

    # 2. แปลงทุกหน้าเป็นรูปภาพแล้ว OCR (เผื่อหน้าที่เป็นไฟล์สแกน)
    print(f'กำลังประมวลผล OCR สำหรับไฟล์ {file_name} (ทุกหน้าเพื่อความสมบูรณ์)...', end='')
    images = convert_from_path(pdf_path)
    print('✅')

    for page_idx, image in enumerate(images):
        page_content = ''
        source_type = ''

        # ถ้า text layer ของหน้านี้มีเนื้อหายาวพอ ใช้อันนั้นเลย (เร็วกว่า แม่นกว่า OCR)
        if page_idx in text_layer_content_by_page and len(text_layer_content_by_page[page_idx]) > 50:
            page_content = text_layer_content_by_page[page_idx]
            source_type = 'text'
        else:
            ocr_text = pytesseract.image_to_string(image, lang='tha+eng')
            if ocr_text.strip():
                page_content = ocr_text.strip()
                source_type = 'ocr'

        if page_content:
            documents.append(
                Document(
                    page_content=page_content,
                    metadata={'page': page_idx, 'source_type': source_type, 'filename': file_name},
                )
            )
    return documents

In [ ]:
# ── 6. โหลดทุกไฟล์ ──
all_documents = []

for pdf_file in pdf_files:
    print(f'กำลังอ่านไฟล์: {pdf_file}...')
    docs = load_single_pdf(pdf_file)
    all_documents.extend(docs)

print(f'\n✅ รวมข้อมูลทั้งหมดเรียบร้อย: {len(all_documents)} หน้า')

In [ ]:
# ── 7. แบ่งเป็น Chunk ──
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=['\n\n', '\n', ' ', ''],
)

chunks = text_splitter.split_documents(all_documents)
for index, chunk in enumerate(chunks):
    chunk.metadata['chunk_id'] = index + 1
print(f'✅ แบ่งข้อมูลเป็น Chunk ทั้งหมด: {len(chunks)} Chunks')

In [ ]:
# ── 8. Embedding Model (BAAI/bge-m3 — ตัวเดียวกับ production) ──
embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'cuda' if __import__('torch').cuda.is_available() else 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)
print('✅ Embedding Model พร้อมใช้งาน')

In [ ]:
# ── 9. สร้าง FAISS Vector Store ──
db = FAISS.from_documents(chunks, embeddings)
db.save_local('faiss_index')
print('✅ สร้างและบันทึก FAISS Vector Store สำเร็จ')

In [ ]:
# ── 10. เรียก Gemini พร้อม Retry เวลา error ──
def generate_answer(prompt):
    retry = 3
    for i in range(retry):
        try:
            return client.models.generate_content(model='gemini-3.6-flash', contents=prompt)
        except Exception as e:
            print(f'⚠️ Gemini Error ครั้งที่ {i+1}: {e}')
            time.sleep(5)
    return None

In [ ]:
# ── 11. RAG Core: ค้นหา + ดึงมาจากไฟล์ที่เกี่ยวข้องที่สุดเพียงไฟล์เดียว + สร้างคำตอบ ──
def ask_chatbot(question, top_k=4, verbose=True):
    greetings = ['สวัสดี', 'hello', 'hi', 'หวัดดี', 'สอบถามครับ', 'สอบถามค่ะ']
    if question.strip().lower() in greetings:
        welcome_msg = 'สวัสดีครับ! มีข้อสงสัยหรือต้องการสอบถามข้อมูลเกี่ยวกับ กยศ. มหาวิทยาลัยสงขลานครินทร์ เรื่องใด สามารถพิมพ์ถามได้เลยครับ'
        if verbose:
            print('\n' + '=' * 60)
            print(f' คำถาม: {question}')
            print('\n AI ตอบ:')
            print(welcome_msg)
            print('=' * 60)
        return welcome_msg

    raw_results = db.similarity_search_with_score(question, k=8)

    if not raw_results:
        if verbose:
            print('ยังไม่มีเอกสารในระบบที่จะค้นหาได้ครับ')
        return ''

    # ดึงมาจากไฟล์ที่เกี่ยวข้องที่สุดเพียงไฟล์เดียว (ไฟล์ของผลลัพธ์อันดับ 1)
    # แทนที่จะปนกันหลายไฟล์ เพื่อให้คำตอบอ้างอิงแหล่งเดียวที่ชัดเจน ไม่สับสน
    best_filename = raw_results[0][0].metadata.get('filename', 'unknown')
    selected_docs = [
        (doc, score) for doc, score in raw_results
        if doc.metadata.get('filename', 'unknown') == best_filename
    ][:top_k]

    if verbose:
        print(f'\n🔍 ดึงข้อมูลจากไฟล์ที่เกี่ยวข้องที่สุด: {best_filename} ({len(selected_docs)} chunks)')

    context = ''
    sources_info = []

    for rank, (doc, _) in enumerate(selected_docs, 1):
        context += doc.page_content + '\n\n'
        page = doc.metadata.get('page', 0) + 1
        filename = doc.metadata.get('filename', 'ไม่ระบุชื่อไฟล์')

        if verbose:
            print(f'[{rank}] 📄 {filename} | หน้า: {page}')

        sources_info.append({'rank': rank, 'filename': filename, 'page': page})

    prompt = f'''คุณคือ AI Chatbot สำหรับตอบคำถามเกี่ยวกับ กองทุนเงินให้กู้ยืมเพื่อการศึกษา (กยศ.)

Context:
{context}

คำถามของผู้ใช้งาน:
{question}

กฎ:
1. ใช้ข้อมูลจาก Context ด้านบนในการตอบเท่านั้น
2. ห้ามสร้างข้อมูลเพิ่มเติมเอง
3. หากไม่มีข้อมูลใน Context ให้ตอบว่า "ไม่พบข้อมูลในเอกสาร"
4. ตอบเป็นภาษาไทย
'''

    response = generate_answer(prompt)

    if verbose:
        print('\n' + '=' * 60)
        print(f' คำถาม: {question}')
        print('\n AI ตอบ:')

    if response:
        ans_text = response.text
        if verbose:
            print(ans_text)

            no_citation_phrases = [
                'ไม่พบข้อมูลในเอกสาร', 'ขออภัย ไม่พบข้อมูลดังกล่าว', 'ยินดีให้บริการ',
                'ยินดีครับ', 'ยินดีค่ะ', 'ด้วยความยินดี', 'พร้อมให้บริการ',
            ]
            show_citation = not any(phrase in ans_text for phrase in no_citation_phrases)

            if show_citation:
                print('\n📄 อ้างอิงแหล่งข้อมูล:')
                for s in sources_info:
                    print(f"[{s['rank']}] 📄 {s['filename']} (หน้า {s['page']})")

            print('=' * 60)
        return ans_text
    else:
        if verbose:
            print('ระบบ AI ไม่สามารถประมวลผลคำตอบได้ในขณะนี้ครับ')
        return ''

In [ ]:
# ── 12. โหมดถาม-ตอบแบบ Interactive ──
while True:
    question = input("\nถามคำถามเกี่ยวกับ กยศ (พิมพ์ 'ออก' เพื่อจบการทำงาน): ")
    if question.strip().lower() in ['exit', 'quit', 'ออก']:
        print('ปิดระบบเรียบร้อยครับ')
        break
    if question.strip():
        ask_chatbot(question, verbose=True)

---
**หมายเหตุ:** โน้ตบุ๊กนี้ใช้ FAISS (in-notebook vector store) และดึงคำตอบจาก **ไฟล์ที่เกี่ยวข้องที่สุดเพียงไฟล์เดียว**
ตรงกับ logic ปัจจุบันของ `ask_rag()` ใน `backend/app/services/rag/rag_engine.py` แล้ว
ส่วนต่างจากระบบจริงมีเพียงจุดเดียว: ระบบจริงเก็บ embedding ใน Postgres ผ่าน pgvector แทน FAISS ในหน่วยความจำ
(เพื่อให้ข้อมูลอยู่ถาวรและรองรับเอกสาร/ประกาศจำนวนมาก) — โน้ตบุ๊กนี้เหมาะสำหรับทดลอง/ปรับจูนแบบเร็วๆ ด้วยไฟล์ตัวอย่างของตัวเอง